# Лаборатория байесовских вычислений: от Метрополиса к PyMC и NUTS
### Дополнительная тетрадь к занятиям 5–6
*Курс: Байесовский анализ эмпирических данных (2026). Автор: Ирина Князева*

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/06_bayesian_computation_ru.ipynb)

English version: [06_bayesian_computation.ipynb](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/06_bayesian_computation.ipynb)

В реальных исследованиях знаменатель теоремы Байеса — маргинальное правдоподобие $p(y) = \int p(y \mid \theta) p(\theta)\, d\theta$ — это невычислимый многомерный интеграл. Аналитически его не взять, а наивный метод отклонения выбрасывает почти все сэмплы.

Эта лаборатория проходит по «лестнице» вычислений современной байесовской статистики:
1. **Прямой Монте-Карло и MCMC:** независимые сэмплы и зависимые марковские цепи.
2. **Метрополис–Гастингс с нуля:** сэмплер ненормированного апостериорного распределения примерно в 15 строк Python.
3. **Размер шага:** почему масштаб предложения — это компромисс между долей принятых шагов и скоростью исследования.
4. **Коррелированный двумерный гребень:** почему случайное блуждание не справляется, когда параметры меняются вместе.
5. **Эталон PyMC/NUTS:** проверка гамильтонова Монте-Карло по точному апостериорному $\operatorname{Beta}(50, 34)$.
6. **Проверки доверия:** $\widehat{R}$, bulk/tail ESS, ранговые графики и расходящиеся переходы — до любой интерпретации.

Страницы книги: [Занятие 5](https://iknyazeva.github.io/bayes-cogsci-book/ru/montecarlotomcmc/) · [Оценивание методом Монте-Карло](https://iknyazeva.github.io/bayes-cogsci-book/ru/montecarloestimation/) · [Метрополис–Гастингс](https://iknyazeva.github.io/bayes-cogsci-book/ru/metropolishastings/) · [HMC и NUTS](https://iknyazeva.github.io/bayes-cogsci-book/ru/hamiltonianmontecarlonuts/) · [Занятие 6](https://iknyazeva.github.io/bayes-cogsci-book/ru/runningsamplertrustchecks/) · [Диагностика сходимости](https://iknyazeva.github.io/bayes-cogsci-book/ru/assessingconvergence/)


In [ ]:
import sys, subprocess
IS_COLAB = "google.colab" in sys.modules
try:
    import pymc as pm
except ImportError:                                   # установить PyMC, если его нет
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pymc"])
    import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import logging, warnings
logging.getLogger("pymc").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.autolayout": True, "axes.grid": True, "grid.alpha": 0.3})
rng = np.random.default_rng(2026)
print("PyMC", pm.__version__, "| ArviZ", az.__version__)


---
## 1. Прямой Монте-Карло: интегрирование моделированием

Если независимые сэмплы $\theta^{(1)}, \dots, \theta^{(S)}$ можно получать прямо из апостериорного распределения $p(\theta \mid y)$, любое апостериорное ожидание $\mathbb{E}[g(\theta) \mid y]$ оценивается выборочным средним:

$$\widehat{\mathbb{E}}[g(\theta)] = \frac{1}{S} \sum_{s=1}^S g(\theta^{(s)}).$$

Численную точность такого конечного моделирования измеряет **стандартная ошибка Монте-Карло (MCSE)**:

$$\operatorname{MCSE}(\bar{g}) = \frac{\operatorname{SD}(g(\theta))}{\sqrt{S}}.$$

> **Апостериорная неопределённость и ошибка Монте-Карло.** Байесовский интервал отражает реальную неопределённость относительно $\theta$ при данных и априорном распределении. MCSE измеряет только шум моделирования: переход от $S$ к $4S$ сэмплам вдвое уменьшает MCSE, но **не** сужает апостериорное распределение.


In [ ]:
# Точный сопряжённый эталон: априорное Beta(2, 2) + 48 из 80 -> Beta(50, 34)
alpha_post, beta_post = 50, 34
exact_mean = alpha_post / (alpha_post + beta_post)          # 50 / 84 = 0.5952
exact_eti = stats.beta.ppf([0.025, 0.975], alpha_post, beta_post)

S = 4000                                                     # независимые сэмплы
direct_draws = rng.beta(alpha_post, beta_post, size=S)
mc_mean = direct_draws.mean()
mcse = direct_draws.std(ddof=1) / np.sqrt(S)

print(f"Точное среднее:          {exact_mean:.5f}")
print(f"Среднее Монте-Карло:     {mc_mean:.5f} ± {mcse:.5f} (MCSE)")
print(f"Точный 95% интервал:     [{exact_eti[0]:.4f}, {exact_eti[1]:.4f}]")
print(f"Квантили по сэмплам:     [{np.percentile(direct_draws, 2.5):.4f}, {np.percentile(direct_draws, 97.5):.4f}]")


---
## 2. Сэмплер Метрополиса–Гастингса с нуля

В сложных моделях независимые сэмплы получить нельзя: знаменатель $p(y)$ неизвестен. Но мы можем вычислить **ненормированное апостериорное распределение**

$$\widetilde{p}(\theta \mid y) = p(y \mid \theta) p(\theta).$$

**Алгоритм Метрополиса–Гастингса** строит марковскую цепь, стационарное распределение которой в точности равно апостериорному:
1. Из текущего состояния $\theta^{(t)}$ предлагаем $\theta^* \sim q(\theta^* \mid \theta^{(t)})$.
2. Вычисляем вероятность принятия
   $$\alpha = \min\left(1, \; \frac{\widetilde{p}(\theta^* \mid y)}{\widetilde{p}(\theta^{(t)} \mid y)} \cdot \frac{q(\theta^{(t)} \mid \theta^*)}{q(\theta^* \mid \theta^{(t)})}\right).$$
   Для симметричного предложения члены с $q$ сокращаются, а неизвестное $p(y)$ сокращается тоже.
3. Сэмплируем $u \sim \operatorname{Uniform}(0, 1)$. Если $u < \alpha$, принимаем $\theta^{(t+1)} = \theta^*$; иначе **остаёмся**: $\theta^{(t+1)} = \theta^{(t)}$.


In [ ]:
def unnormalized_log_posterior(theta, y=48, n=80, a_prior=2, b_prior=2):
    """Логарифм совместной плотности log p(y, theta) = log p(y | theta) + log p(theta)."""
    if theta <= 0.0 or theta >= 1.0:
        return -np.inf                                      # вне пространства параметров
    return stats.beta.logpdf(theta, a_prior, b_prior) + stats.binom.logpmf(y, n, theta)


def run_metropolis(log_target, theta_start, n_draws, prop_sd, seed=2026):
    """Сэмплер Метрополиса со случайным блужданием."""
    gen = np.random.default_rng(seed)
    chain = np.zeros(n_draws)
    current, current_logp = theta_start, log_target(theta_start)
    accepted = 0
    for t in range(n_draws):
        proposal = current + gen.normal(0, prop_sd)         # гауссово предложение
        proposal_logp = log_target(proposal)
        if np.log(gen.uniform()) < proposal_logp - current_logp:   # логарифм отношения
            current, current_logp = proposal, proposal_logp
            accepted += 1
        chain[t] = current                                  # отказ: повторяем текущее состояние
    return chain, accepted / n_draws


chain_test, acc_test = run_metropolis(unnormalized_log_posterior, 0.5, 5000, 0.15)
print(f"Пробный запуск: принято {acc_test:.0%}, среднее после разогрева {chain_test[500:].mean():.4f} (точно {exact_mean:.4f})")


---
## 3. Эксперимент с размером шага

В алгоритме Метрополиса со случайным блужданием эффективность определяет стандартное отклонение предложения $\sigma_{\text{prop}}$:
* **Слишком мал ($\sigma = 0.01$):** предложения принимаются почти всегда, но цепь исследует пространство медленной диффузией.
* **Слишком велик ($\sigma = 2.50$):** большинство предложений попадает в области очень малой плотности; цепь отказывается и «залипает».
* **Подобран ($\sigma = 0.15$):** баланс движения и принятия (для одномерной цели, похожей на нормальную, эффективна доля принятия около 44%).

**Сначала предскажите:** у какой цепи будет наибольшая доля принятия? У какой автокорреляция затухает быстрее всего?


In [ ]:
N_DRAWS = 5000
scales = [0.01, 0.15, 2.5]
labels = ["Слишком мал (σ = 0.01)", "Подобран (σ = 0.15)", "Слишком велик (σ = 2.50)"]
colors = ["tab:orange", "tab:blue", "tab:red"]
results = [run_metropolis(unnormalized_log_posterior, 0.1, N_DRAWS, s) for s in scales]

fig, axes = plt.subplots(3, 2, figsize=(13, 8))
for i, ((chain, acc), label, col) in enumerate(zip(results, labels, colors)):
    ess = float(np.asarray(az.ess(chain[None, 500:])))
    axes[i, 0].plot(chain[:1000], lw=0.8, color=col)
    axes[i, 0].axhline(exact_mean, color="black", ls="--", lw=1, alpha=0.7)
    axes[i, 0].set_title(f"{label}: принято {acc:.0%}, ESS ≈ {ess:.0f}")
    axes[i, 0].set_ylabel(r"$\theta$")
    axes[i, 0].set_ylim(0, 1)
    acf = [1.0] + [np.corrcoef(chain[500:-lag], chain[500 + lag:])[0, 1] for lag in range(1, 40)]
    axes[i, 1].stem(range(40), acf, basefmt=" ")
    axes[i, 1].set_title("Автокорреляция (лаги 0–39)")
    axes[i, 1].set_ylim(-0.1, 1.05)
axes[2, 0].set_xlabel("Итерация")
axes[2, 1].set_xlabel("Лаг")
plt.show()


---
## 4. Коррелированный двумерный гребень: почему случайное блуждание не справляется

Когда параметров больше, они часто коррелированы (свободный член и наклон, компоненты дисперсии). Случайное блуждание предлагает шаги в круглом шаре, поэтому большинство предложений пересекает узкий диагональный гребень и отклоняется.

Это мотивирует **гамильтонов Монте-Карло (HMC)**: он использует градиент $\log p(\theta \mid y)$, чтобы скользить вдоль гребня, — «скейтбордист» из занятия 5 ([HMC и NUTS](https://iknyazeva.github.io/bayes-cogsci-book/ru/hamiltonianmontecarlonuts/)).


In [ ]:
# Коррелированный двумерный нормальный гребень (корреляция 0.92)
mu_2d = np.array([0.0, 0.0])
cov_2d = np.array([[1.0, 0.92], [0.92, 1.0]])
inv_cov_2d = np.linalg.inv(cov_2d)
log_target_2d = lambda theta: -0.5 * theta @ inv_cov_2d @ theta

n_steps, prop_sd_2d = 1500, 0.25
rw_chain = np.zeros((n_steps, 2))
curr = np.array([-2.0, -2.0])
for t in range(n_steps):
    prop = curr + rng.normal(0, prop_sd_2d, size=2)
    if np.log(rng.uniform()) < log_target_2d(prop) - log_target_2d(curr):
        curr = prop
    rw_chain[t] = curr

x_grid, y_grid = np.meshgrid(np.linspace(-3, 3, 100), np.linspace(-3, 3, 100))
z_density = stats.multivariate_normal(mu_2d, cov_2d).pdf(np.dstack((x_grid, y_grid)))
plt.figure(figsize=(7, 6))
plt.contour(x_grid, y_grid, z_density, levels=7, cmap="Blues_r", alpha=0.7)
plt.plot(rw_chain[:400, 0], rw_chain[:400, 1], color="crimson", alpha=0.6, lw=0.9, marker=".", markersize=3, label="случайное блуждание (400 шагов)")
plt.scatter(*rw_chain[0], color="black", s=80, zorder=5, label="старт (-2, -2)")
plt.title("Случайное блуждание на коррелированном гребне (ρ = 0.92)")
plt.xlabel(r"$\theta_1$"); plt.ylabel(r"$\theta_2$"); plt.legend()
plt.show()


---
## 5. Эталон PyMC и NUTS

Теперь рабочий инструмент: **PyMC** с сэмплером **NUTS** (No-U-Turn Sampler). Подгоняем ту же бета-биномиальную задачу ($y = 48$, $n = 80$, априорное Beta(2, 2)), точный ответ которой — $\operatorname{Beta}(50, 34)$ — нам известен.


In [ ]:
with pm.Model() as benchmark_model:
    theta = pm.Beta("theta", alpha=2, beta=2)                      # априорное
    pm.Binomial("y_obs", n=80, p=theta, observed=48)               # правдоподобие: 48 из 80
    idata = pm.sample(draws=1000, tune=1000, chains=4, random_seed=2026, progressbar=False)
print("Сэмплирование завершено")


---
## 6. Проверки доверия

Прежде чем интерпретировать модель, пройдите проверки курса:

| Диагностика | Правило курса | Что проверяет |
|:---|:---|:---|
| $\widehat{R}$ | $\widehat{R} \le 1.01$ | Исследуют ли цепи одно и то же распределение? |
| Bulk ESS | $\ge 400$ | Достаточно ли эффективных сэмплов для средних и медиан? |
| Tail ESS | $\ge 400$ | Устойчивы ли численно концы интервала (2,5%, 97,5%)? |
| Расходящиеся переходы | ровно 0 | Встретил ли сэмплер области, которые не смог пройти? |
| Ранговые графики | примерно равномерные для каждой цепи | Хорошо ли перемешаны цепи по всему носителю? |


In [ ]:
print(az.summary(idata, var_names=["theta"]))

r_hat = float(np.asarray(az.rhat(idata, var_names=["theta"])["theta"]))
ess_bulk = float(np.asarray(az.ess(idata, var_names=["theta"])["theta"]))
ess_tail = float(np.asarray(az.ess(idata, var_names=["theta"], method="tail")["theta"]))
n_divergences = int(np.asarray(idata.sample_stats["diverging"]).sum())
print(f"\nR-hat {r_hat:.3f} | bulk ESS {ess_bulk:.0f} | tail ESS {ess_tail:.0f} | расходящихся переходов {n_divergences}")

az.plot_rank(idata, var_names=["theta"])
plt.show()


### Сверка NUTS с точным ответом

In [ ]:
posterior_theta = idata.posterior["theta"].values.ravel()        # 4000 сэмплов всех цепей
x_vals = np.linspace(0.35, 0.85, 300)
plt.figure(figsize=(9, 4.5))
plt.hist(posterior_theta, bins=40, density=True, alpha=0.5, color="teal", label=f"PyMC NUTS, среднее {posterior_theta.mean():.4f}")
plt.plot(x_vals, stats.beta.pdf(x_vals, alpha_post, beta_post), color="darkred", lw=2.2, label=f"точное Beta(50, 34), среднее {exact_mean:.4f}")
plt.axvline(exact_mean, color="darkred", ls="--", lw=1)
plt.title("PyMC NUTS воспроизводит точное апостериорное распределение")
plt.xlabel(r"$\theta$ (вероятность успеха)"); plt.ylabel("плотность"); plt.legend()
plt.show()


---
## 7. Практика и размышление

### Задание: ошибка Монте-Карло для конкретной величины
Вычислите по сэмплам PyMC апостериорную вероятность того, что $\theta > 0.65$,

$$p = \Pr(\theta > 0.65 \mid y) = \mathbb{E}[\mathbb{I}(\theta > 0.65) \mid y],$$

и её стандартную ошибку Монте-Карло по дисперсии индикатора и tail ESS:

$$\operatorname{MCSE}(p) \approx \sqrt{\frac{p(1 - p)}{\text{ESS}_{\text{tail}}}}.$$


In [ ]:
indicator_draws = (posterior_theta > 0.65).astype(float)
prob_est = indicator_draws.mean()
exact_prob = stats.beta.sf(0.65, alpha_post, beta_post)
mcse_prob = np.sqrt(prob_est * (1.0 - prob_est) / ess_tail)

print("Цель: Pr(θ > 0.65 | y)")
print(f"Точная вероятность:   {exact_prob:.4f}")
print(f"Оценка PyMC:          {prob_est:.4f} ± {mcse_prob:.4f} (MCSE)")
print(f"Абсолютная ошибка:    {abs(prob_est - exact_prob):.4f}")


### Вопросы для размышления

1. **Почему высокая доля принятия не означает эффективного сэмплирования?** В разделе 3 цепь с $\sigma = 0.01$ принимала почти все предложения, но её автокорреляция затухала очень медленно, а ESS был мал. Почему?
2. **Зачем нужны четыре цепи?** Может ли одна цепь выявить многомодальность или отсутствие сходимости через $\widehat{R}$?
3. **Что означает хотя бы одно расхождение?** Подгонка сообщает $\widehat{R} = 1.00$ и большой ESS, но 3 расходящихся перехода. Можно ли интерпретировать апостериорное распределение?
